<a href="https://colab.research.google.com/github/sureshaboutula/Data_Engineering/blob/main/confluent_kafka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Produce a message to Confluent Kafka Topic via Python

In [ ]:
# Required connection configs for Kafka producer, consumer, and admin
bootstrap.servers=pkc-619z3.us-east1.gcp.confluent.cloud:9092
security.protocol=SASL_SSL
sasl.mechanisms=PLAIN
sasl.username=74HP6X44RTXY7QAV
sasl.password=Vme0+ZG0l5JcCiMNfZbBbCNqUJpkMNxpMqmszsTZteSjPHcpl0Wlx2OvJfUf7RhS

# Best practice for higher availability in librdkafka clients prior to 1.7
session.timeout.ms=45000

client.id=ccloud-python-client-e6ad6149-d4e5-4b52-89ad-963c1ef98174


In [1]:
!pip install confluent-kafka

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 48.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import json

csv_file = '/home/Customer_Data/first_100_customers.csv'
df = pd.read_csv(csv_file)
df.head()

,customer_id,name,city,state,country,registration_date,is_active
0,0,Customer_0,Pune,Maharashtra,India,2023-06-29,False
1,1,Customer_1,Bangalore,Tamil Nadu,India,2023-12-07,True
2,2,Customer_2,Hyderabad,Gujarat,India,2023-10-27,True
3,3,Customer_3,Bangalore,Karnataka,India,2023-10-17,False
4,4,Customer_4,Ahmedabad,Karnataka,India,2023-03-14,False


In [3]:
# Convert CSV to JSON
json_records = df.to_dict(orient='records')
json_file = '/home/Customer_Data/customers.json'

with open(json_file, 'w') as file:
  json.dump(json_records, file, indent=4)

print(f"CSV file '{csv_file}' has been converted to JSON and saved as '{json_file}'.")


CSV file '/home/Customer_Data/first_100_customers.csv' has been converted to JSON and saved as '/home/Customer_Data/customers.json'.


In [6]:
from confluent_kafka import Producer
import json
import time

config = {
"bootstrap.servers":"pkc-619z3.us-east1.gcp.confluent.cloud:9092",
"security.protocol":"SASL_SSL",
"sasl.mechanisms":"PLAIN",
"sasl.username":"74HP6X44RTXY7QAV",
"sasl.password":"Vme0+ZG0l5JcCiMNfZbBbCNqUJpkMNxpMqmszsTZteSjPHcpl0Wlx2OvJfUf7RhS",
"session.timeout.ms":45000,
"client.id":"ccloud-python-client-e6ad6149-d4e5-4b52-89ad-963c1ef98174 "
}

#Create a Producer
producer = Producer(config)

In [10]:
topic = 'ecommerce'

with open('/home/Customer_Data/customers.json', 'r') as file:
  custoemrs_data = json.load(file)

value = custoemrs_data[0]
key = value['customer_id']
print(key,value)

0 {'customer_id': 0, 'name': 'Customer_0', 'city': 'Pune', 'state': 'Maharashtra', 'country': 'India', 'registration_date': '2023-06-29', 'is_active': False}


In [11]:
producer.produce(topic, key=str(key).encode('utf-8'), value=str(value).encode('utf-8'))

In [ ]:
# Send multiple messages to Kafka


In [13]:
def delivery_status(err, msg):
  if err:
    print(f"Message delivery failed: {err}")
  else:
    print(f"Message delivered to {msg.topic()} [{msg.partition()}] at offset {msg.offset()}")

for record in custoemrs_data:
  try:
    message_value = json.dumps(record)
    message_key = str(record['customer_id']).encode('utf-8')
    producer.produce(topic, key=message_key, value=message_value, callback = delivery_status)
    producer.poll(1)
  except Exception as e:
    print(f"Error sending message: {e}")

producer.flush()

print("All messages sent to Kafka successfully.")

Message delivered to ecommerce [2] at offset 3
Message delivered to ecommerce [2] at offset 4
Message delivered to ecommerce [1] at offset 0
Message delivered to ecommerce [1] at offset 1
Message delivered to ecommerce [1] at offset 2
Message delivered to ecommerce [1] at offset 3
Message delivered to ecommerce [1] at offset 4
Message delivered to ecommerce [0] at offset 1
Message delivered to ecommerce [2] at offset 5
Message delivered to ecommerce [0] at offset 2
Message delivered to ecommerce [0] at offset 3
Message delivered to ecommerce [0] at offset 4
Message delivered to ecommerce [0] at offset 5
Message delivered to ecommerce [2] at offset 6
Message delivered to ecommerce [0] at offset 6
Message delivered to ecommerce [1] at offset 5
Message delivered to ecommerce [0] at offset 7
Message delivered to ecommerce [2] at offset 7
Message delivered to ecommerce [0] at offset 8
Message delivered to ecommerce [1] at offset 6
Message delivered to ecommerce [0] at offset 9
Message deliv